# Lab 2: Linear Regression

**CS 463 · Due Wednesday, Oct 7, 11:59 PM in Canvas**

Based on §3.6 (Lab: Linear Regression) of James, Witten, Hastie, Tibshirani, and Taylor,
*An Introduction to Statistical Learning with Applications in Python* (Springer, 2023),
using the `Carseats` data from the `ISLP` package.

### Before you start: install `plotly`

This lab uses `plotly` for its rotatable 3-D plots, and your `CS_463` environment probably doesn't
have it yet. If the imports cell below fails with `ModuleNotFoundError: No module named 'plotly'`,
install it using whichever setup you have, then **restart the kernel** and run the imports cell
again. (On Colab, there's nothing to do: `plotly` is already installed.)

<details>
<summary>If you use a virtual environment (<code>venv</code>)</summary>

In a terminal, activate your environment, then install. Use your environment's folder name in
place of `CS_463`.

macOS / Linux:

```bash
source CS_463/bin/activate
pip install plotly nbformat
```

Windows (PowerShell):

```powershell
CS_463\Scripts\Activate.ps1
pip install plotly nbformat
```

</details>

<details>
<summary>If you use conda</summary>

In a terminal (on Windows, the Anaconda Prompt), activate your environment, then install. Use
your environment's name in place of `CS_463`.

```bash
conda activate CS_463
conda install -c conda-forge plotly nbformat
```

</details>

### How this lab works

- Work through **Parts 1–6** in order. Parts marked *(awareness)* (2, 4, and 6) are reading only:
  run the cells and read the explanations. There's nothing to turn in for them.
- Everything you turn in is labeled by part number and letter, like **3(c)**, and marked as
  either **Code** or **Written**:
  - **Code:** write your code in the cell right below, which starts with `# 3(b)` and
    `# YOUR CODE HERE`.
  - **Written:** replace `YOUR ANSWER HERE` in the **3(c) Answer:** cell right below. Written
    answers: 1–3 sentences.
- Items marked *(optional)* aren't graded.
- Anything else, like "Run this cell," is just an instruction. There's nothing to turn in.

**Checklist** (15 required items):

| Part | Code | Written |
|---|---|---|
| 1 | 1(a) | 1(b), 1(c), 1(d), 1(e) |
| 3 | 3(a), 3(b), 3(d) | 3(c), 3(e), 3(f), 3(g) |
| 5 | 5(a) | 5(b), 5(c) (and optional 5(d)) |

- Keep `ch_03_lab_helpers.py` in the same folder as this notebook. It holds the plotting code, and you don't need to read it. Don't turn it in with your submission (I have my own copy).
- Before submitting, run **Restart & Run All** and make sure every cell runs.

**Note:** If you just run everything right now, you'll get some errors because I've provided some cells that depend on you to complete previous cells. This isn't anything to worry about. Just letting you know why you might get certain errors.

**What you're responsible for.** You're responsible for **interpreting multiple regression
coefficients** and **dummy (categorical) variables**. For now, correlation, the F-statistic,
collinearity/VIF, and correlated errors are covered at an **awareness level**: know what each
one means and doesn't mean. I'll give you an update well before I quiz you on any of the
awareness-level content.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from ISLP import load_data

from ch_03_lab_helpers import (compare_coefficient, plot_correlation_matrix,
                               plot_correlation_pitfalls, plot_correlation_strength,
                               plot_fit, plot_parallel_lines, plot_plane, plot_slices)

## The data: Carseats

A simulated data set of child car seat sales at 400 stores. The columns we use:

| Column | Meaning |
|---|---|
| `Sales` | unit sales at the store, in **thousands** |
| `Price` | the price the store charges for a car seat (the docs give no units; we'll read it as dollars) |
| `CompPrice` | the price a competitor charges at the same location |
| `Advertising` | the store's local advertising budget, in **thousands of dollars** |
| `Age` | average age of the local population, in years |
| `US` | whether the store is in the US: `No` or `Yes` |
| `ShelveLoc` | quality of the store's shelf location for car seats: `Bad`, `Medium`, or `Good` |

In [ ]:
Carseats = load_data('Carseats')
Carseats.head()

## Part 1: Simple linear regression

In class, we fit lines to the `Advertising` data step by step. Here's the same process with real
software, predicting `Sales` from `Price`. The code is given; your job is to read and interpret
the output.

We fit models with `statsmodels`, the standard Python library for regression output with
standard errors and p-values. `smf.ols('Sales ~ Price', data=Carseats)` sets up a so-called "ordinary least squares" (OLS) model. Read `~` as "is modeled by": the response goes on the left, and the predictors go on the right, joined with `+` when there's more than one. `.fit()` finds the least squares coefficients and returns the fitted model.

Run these cells to fit the model, plot the data with its least squares line, and print
the model summary. *(You don't need to understand how the plot was generated.)*

<details>
<summary>"ordinary" vs "simple": regression vocabulary?</summary>

**Ordinary** means the plain version of least squares we used in class: minimize RSS, with every
observation counting equally. Variants change that. For example, *weighted* least squares lets
some observations count more than others, and *generalized* least squares allows for correlated
errors (Part 6).

**Simple** is about something else: the number of predictors. Simple regression has one
predictor; multiple regression has more than one. OLS can be either. We use `smf.ols` for the
simple model here and for the multiple regression models in Part 3.

</details>

In [ ]:
price_model = smf.ols('Sales ~ Price', data=Carseats).fit()
plot_fit(Carseats, 'Price', 'Sales', price_model);

In [ ]:
price_model.summary()

**Reading the summary.** Most of what we covered in class is in this table. 

| In class | In the summary | As code |
|---|---|---|
| slope and intercept | `coef` | `price_model.params` |
| standard error (SE) | `std err` | `price_model.bse` |
| t-statistic | `t` | `price_model.tvalues` |
| p-value | `P>\|t\|` | `price_model.pvalues` |
| 95% confidence interval | `[0.025  0.975]` | `price_model.conf_int()` |
| $R^2$ | `R-squared` (top right) | `price_model.rsquared` |
| RSS | not shown | `price_model.ssr` |
| RSE (the noise sd) | not shown | `np.sqrt(price_model.scale)` |

You can ignore the rest of the summary for now. (`ssr` stands for "sum of squared residuals",
which is just another, vastly superior name for RSS.) 

In the table above, each of the first five attributes (or methods) in the right-most column holds (or computes) a value for every coefficient; for example, `price_model.params['Price']` is the slope. The last three are single numbers that describe the whole model.

**1(a) · Code.** Using the attributes in the table above:

1. Compute the t-statistic for `Price` (really, the coefficient of `Price` in the model, which is the slope) yourself, as estimate ÷ SE, and check that it matches the summary.
2. Compute estimate ± 2 SE, and compare it with the summary's 95% confidence interval.
3. Print $R^2$, the RSS, and the RSE.

In [ ]:
# 1(a)
### YOUR CODE HERE


**1(b) · Written.** Interpret the `Price` slope in one sentence, with units. (Remember that `Sales` is in
thousands of units.)

**1(b) Answer:**

_YOUR ANSWER HERE_


**1(c) · Written.** The intercept is the model's predicted sales at a price of \$0. Is that a meaningful
prediction here? Look at the range of prices in the plot.

**1(c) Answer:**

_YOUR ANSWER HERE_


**1(d) · Written.** What does the 95% confidence interval for the slope tell you? What do the t-statistic
and p-value tell you?

**1(d) Answer:**

_YOUR ANSWER HERE_


**1(e) · Written.** Interpret $R^2$ and the RSE in words. $R^2$ is only about 0.2. Does that mean price
doesn't matter?

**1(e) Answer:**

_YOUR ANSWER HERE_


## Part 2: Correlation *(awareness)*

**Pearson's Correlation** ($r$) is a number from −1 to 1 that measures how strongly two variables move
together **in a straight line**. We haven't covered this in class, yet. We'll often just call this **correlation**, though technically there are other ways to measure this concept. If $r$ is near ±1, the points hug a line; near 0, there's no linear
pattern.

- **The sign is the direction.** Positive: $y$ tends to rise as $x$ rises. Negative: $y$ tends to
  fall as $x$ rises, like `Sales` and `Price` in Part 1.
- **The size is the strength:** how tightly the points hug a straight line.

In [ ]:
# Simulated data. You don't need to understand how these plots were generated.
plot_correlation_strength();

What $r$ does **not** tell you:

- **Not causation.** Two variables can move together because something else drives both (ice cream sales and shark attacks are correlated! Does that mean ice cream sales cause shark attacks? No!).
- **Not a slope.** $r$ measures how tightly the points hug a line relative to their overall spread, not how steep the line is. (The two do share a sign: a positive $r$ means an upward-sloping line.) 
- **Not useful for ruling out *some other* relationship.** Third panel: $y$ depends strongly on $x$, yet $r$ is near 0.
- **Sensitive to outliers.** Fourth panel: 30 points with no relationship, plus one extreme
  point, give a sizable $r$. Always look at the plot, not just the number.

In [ ]:
plot_correlation_pitfalls();

**Above:** $r$ can't distinguish a steep line from a shallow line. It can't tell you if there's really *some other* relationship than linear, or whether there only *seems* to be a relationship because of the influence of outliers.

**Connection to $R^2$.** In *simple* regression, our model only has one independent variable and it turns out that $R^2 = r^2$. This is not true in multiple regression (many independent variables, so we're fitting planes and hyper planes), which we cover later in the lab. Below, we check this with Part 1's model.         

In [ ]:
r = Carseats['Sales'].corr(Carseats['Price'])
print(f"r = {r:.4f},  r² = {r**2:.4f},  R² from Part 1 = {price_model.rsquared:.4f}")

That's only true with one predictor, and even then $r$ tells you one extra thing: its sign gives the direction of the relationship ($r$ is negative here because sales fall as price rises; $R^2$ can't show that). More importantly, $r$ describes any two variables, while $R^2$ describes how well a fitted model predicts its response. That's why, in Part 3, we'll use $r$ to see how *predictors relate to each other*. With several predictors, $R^2$ measures how well they explain the response together, so it isn't the square of any single predictor's correlation with the response.

## Part 3: Multiple regression

### "Holding the other predictors fixed"

Multiple regression uses several predictors at once. With two:

$$
\text{Sales} = \beta_0 + \beta_1 \cdot \text{Advertising} + \beta_2 \cdot \text{Age} + \text{error}
$$

(ISL writes coefficients as $\beta$ s. In class, we called the slope $m$ and the intercept $b$.)
Least squares still picks the coefficients that make RSS as small as possible.

**In Code:** To use more than one predictor, list them all on the right of `~`,
joined with `+`:

```python
smf.ols('Sales ~ Advertising + Age', data=Carseats).fit()
```

- In a formula, `+` means "also include this predictor". It's not summing anything.
- `statsmodels` automatically generates the intercept ($\beta_0$) for you, so this model has three coefficients:
  `Intercept`, `Advertising`, and `Age`.
- The order of the predictors doesn't change the fit. For more predictors, include them with additional `+`-signs:
  `'Sales ~ Advertising + Age + Price'`.

In [ ]:
ad_age_model = smf.ols('Sales ~ Advertising + Age', data=Carseats).fit()
ad_age_model.params

With two predictors, the fit is a **plane** instead of a line. Drag the plot to rotate it.
*(You don't need to understand how this plot was generated.)*

In [ ]:
plot_plane(Carseats, 'Advertising', 'Age', 'Sales', ad_age_model)

### Making predictions

To predict sales for a store, plug its predictor values into the fitted equation:

$$\hat{y} = \hat\beta_0 + \hat\beta_1 \cdot \texttt{Advertising} + \hat\beta_2 \cdot \texttt{Age}$$

where the $\hat\beta$'s are the numbers in `ad_age_model.params`. Geometrically, the prediction is
the height of the plane above that store's (`Advertising`, `Age`) point on the floor.

**3(a) · Code.** Predict sales for a store with an advertising budget of 10 (\$10,000) and a
local population whose average age is 50. Use the code cell as your calculator, showing each step:

1. Pull each coefficient out of `ad_age_model.params` by name (for example,
   `ad_age_model.params['Age']`) and store it in its own variable.
2. Write out the equation above with those variables and the store's values, and display the
   result.

Remember that `Sales` is in thousands of units. You'll check your answer in 3(b).

In [ ]:
# 3(a)
### YOUR CODE HERE


**3(b) · Code.** In practice, you let the model do that arithmetic. The `predict` method takes a
DataFrame of stores, with one row per store and one column per predictor. The column names must
match the predictor names in the formula exactly. It returns one prediction per row, in order:

```python
example_stores = pd.DataFrame({'Advertising': [5, 15],
                               'Age':         [40, 60]})
example_preds = ad_age_model.predict(example_stores)
example_preds.iloc[0]   # the first store's prediction (.iloc[1] is the second's)
```

Now consider two stores with the same advertising budget (10), whose local populations have
average ages of 50 (store A) and 30 (store B). Use `predict` to:

1. Get both stores' predictions with one call to `predict`, and check that store A's matches your 3(a) answer.
2. Compute how much higher store B's predicted sales are than store A's (B − A).

In [ ]:
# 3(b)
### YOUR CODE HERE


Each coefficient is the slope **along its own axis, with the other predictor held fixed**.
Slicing the plane at a few fixed ages gives parallel lines: whatever the age, one more unit of
`Advertising` moves the prediction by the same amount, $\beta_1$. Each store is colored like the slice its age is closest to.

In [ ]:
plot_slices(Carseats, 'Advertising', 'Age', 'Sales', ad_age_model, values=[30, 50, 70]);

So, **holding `Age` fixed**, each extra \$1,000 of advertising goes with about 0.114 thousand,
or 114, more units sold, on average. **Holding `Advertising` fixed**, each extra year of average
age goes with about 40 fewer units sold.

With more than two predictors, we can't draw the fit: it lives in more dimensions than we can
picture. The math is the same, though: one coefficient per predictor, each one the slope along
its own axis with all the others held fixed.

### Worked example: uncorrelated predictors

Does adding a predictor change the other predictors' coefficients? Let's compare each
predictor's coefficient on its own with its coefficient in the combined model.

In [ ]:
ad_model = smf.ols('Sales ~ Advertising', data=Carseats).fit()
age_model = smf.ols('Sales ~ Age', data=Carseats).fit()

display(compare_coefficient({'Advertising alone': ad_model, 'Advertising + Age': ad_age_model}, 'Advertising'))
display(compare_coefficient({'Age alone': age_model, 'Advertising + Age': ad_age_model}, 'Age'))
print(f"correlation between Advertising and Age: {Carseats['Advertising'].corr(Carseats['Age']):.3f}")

**Above:** In each table, the top row corresponds to a simple regression model (one predictor) and the second row corresponds to a multiple regression (in this case, 2 predictors) model.

Both coefficients barely move: `Advertising` stays at about 0.114, and `Age` stays at about
−0.040. Unlike shark attacks and ice cream sales, the two predictors (advertising and age) are essentially uncorrelated ($r \approx -0.005$), so knowing the average age of the population near a store tells you nothing about its advertising budget. We'll soon see examples of using correlated predictors.

**3(c) · Written.** In your own words: when two predictors are uncorrelated, what happens to each one's
coefficient when the other is added to the model? Why do you think this is so?

*Hint: In the next section, you work with correlated predictors. You might want to revisit this question after you work through it.*

**3(c) Answer:**

_YOUR ANSWER HERE_


### Your turn: correlated predictors

Now let's look at `CompPrice`, the price a competitor charges near each store.

**3(d) · Code.** Fit `Sales ~ CompPrice` and name it `compprice_model`. Then fit `Sales ~ CompPrice + Price`
and name it `both_model`. Use those names, because a later cell refers to them.

In [ ]:
# 3(d)
### YOUR CODE HERE


Run this cell to compare the `CompPrice` coefficient between your two models.

In [ ]:
compare_coefficient({'CompPrice alone': compprice_model, 'CompPrice + Price': both_model}, 'CompPrice')

**3(e) · Written.** How did the `CompPrice` coefficient and its p-value change when `Price` was added? Based on
the simple model alone, would you have concluded that the competitor's price matters?

**3(e) Answer:**

_YOUR ANSWER HERE_


**3(f) · Written.** Interpret the `CompPrice` coefficient from `both_model` in one sentence, with units,
using the phrase "holding ... fixed".

**3(f) Answer:**

_YOUR ANSWER HERE_


Here's the correlation matrix of the variables from the worked example and this section. *(You don't need to understand
how this plot was generated.)*

In [ ]:
plot_correlation_matrix(Carseats, ['Sales', 'CompPrice', 'Price', 'Advertising', 'Age']);

In future labs/assignments, you'll learn to create these yourself. Notice that CompPrice and Price are moderately correlated with each other, unlike any other pair of predictors. 

**Seeing it in 3-D.** Here's the fitted plane for your `both_model`. Rotate it so you're looking
straight down at the `CompPrice`–`Price` floor, and compare with the Advertising-Age plane in the previous 3-D plot. *(You don't
need to understand how these plots were generated.)*

In [ ]:
plot_plane(Carseats, 'CompPrice', 'Price', 'Sales', both_model)

From "above" (that is, the high Sales value direction), the stores form a (somewhat noisy) diagonal band: stores whose competitors charge
more tend to charge more themselves. That's the correlation ($r \approx 0.58$) in the matrix, and
it means the two predictors carry **overlapping information**. Knowing one tells you something
about the other.

In the simple model, `CompPrice` has to stand in for the information it shares with `Price`. So
its coefficient mixes two things: the competitor's price, which goes with higher sales, and the
higher own price that tends to come with it, which goes with lower sales. They are in tension.

With both predictors in the model, least squares can separate the two effects, because the
overlap is only partial. Later in the lab, we'll see how stronger correlation between predictors causes problems.

<details>
<summary>Optional: the arithmetic</summary>

On average, a store whose competitor charges \$1 more charges about \$0.90 more itself. Holding
`Price` fixed, that extra \$1 of competitor price goes with about 91 more units sold, and the
extra \$0.90 of own price goes with about $0.90 \times 87 \approx 79$ fewer. The net is
$91 - 79 = 12$ units: the simple model's slope.

</details>

**3(g) · Written.** In your own words, explain why adding `Price` changed the `CompPrice` coefficient so
much, while adding `Age` barely changed the `Advertising` coefficient in the worked example. Use the idea of
overlapping information.

**3(g) Answer:**

_YOUR ANSWER HERE_


## Part 4: More of the regression table *(awareness)*

### The F-statistic

Every summary reports an `F-statistic` and its (the F-statistic's) p-value, `Prob (F-statistic)`, in the top right.
The F-test asks whether **at least one** predictor in the model is useful for predicting the
response. It doesn't tell you **which** one. It exists because, with many predictors, some
individual p-values will look small just by chance; the F-test checks the model as a whole.

In [ ]:
print(f"Advertising + Age:  F = {ad_age_model.fvalue:.1f},  p-value = {ad_age_model.f_pvalue:.1e}")

**Reading these numbers.** If neither `Advertising` nor `Age` had any real relationship with
`Sales`, $F$ would typically land near 1. Here it's 28.6, and the p-value ($2.6 \times 10^{-12}$)
says an $F$ this large would essentially never happen by chance if both predictors were useless
for making `Sales` predictions. So at least one of them is useful.

The F-test alone can't say **which** one. For that, look at each predictor's own t-test. Here's
the coefficient table from `ad_age_model`'s summary:

In [ ]:
ad_age_model.summary().tables[1]

The `P>|t|` column holds each coefficient's p-value. Both are tiny (`Advertising` about
$2 \times 10^{-8}$, `Age` about $10^{-6}$; the table rounds them to 0.000), so each predictor is
useful even with the other held fixed.

With only one predictor, the F-test's question ("is at least one predictor useful?") and the
t-test's question ("is `Price` useful?") are the same question, so the two tests agree exactly:

In [ ]:
# With a single predictor, the F-test and that predictor's t-test give the same p-value
print(f"Price alone:  F-test p-value = {price_model.f_pvalue:.2e},  t-test p-value = {price_model.pvalues['Price']:.2e}")

**So why do we need the F-test at all?** Why not just check each predictor's p-value, as we
just did? Because each t-test has its own chance of a false alarm. With a 0.05 cutoff, a
predictor that has nothing to do with the response still gets $p < 0.05$ about 5% of the time.
With two predictors that's a small risk. With 100 useless predictors, you'd expect about 5 of
them to look "significant" purely by chance. The F-test is a **single** test of the whole model,
so its false-alarm rate stays at 5% no matter how many predictors there are.

To see this, the next cell fits `Sales` on 100 columns of pure random noise. None of them can
possibly predict sales. *(You don't need to understand how the noise is generated.)*

In [ ]:
rng = np.random.default_rng(464)
noise = pd.DataFrame(rng.normal(size=(len(Carseats), 100)),
                     columns=[f'noise{i}' for i in range(1, 101)])
noise['Sales'] = Carseats['Sales'].to_numpy()
noise_model = smf.ols('Sales ~ ' + ' + '.join(noise.columns[:-1]), data=noise).fit()

noise_pvalues = noise_model.pvalues.drop('Intercept')
print(f"noise predictors with p < 0.05: {(noise_pvalues < 0.05).sum()} of 100")
print(f"F-test p-value: {noise_model.f_pvalue:.2f}")

Checked one at a time, 6 of the 100 noise columns look like useful predictors, which is exactly
the kind of false alarm described above. The F-test, asking about all 100 at once, gives a
p-value of 0.88: no evidence that any of them is useful, which is the right answer.

So a more reliable, typical procedure is: check the F-test first to see whether the model as a whole is useful,
then look at individual p-values to see which predictors are doing the work. With one or two
predictors the F-test adds little, but with many predictors it guards against being fooled by
chance.

### Collinearity and VIF

In Part 3, `CompPrice` and `Price` overlapped only partly, so least squares could still separate
their effects reliably. When the overlap is nearly total, it can't. **Collinearity** means two or
more predictors are highly correlated with each other. `Carseats`
doesn't have any, so we'll borrow ISL's `Credit` data: credit card `Balance` for 400 people,
along with their credit `Limit` and credit `Rating`, which "move together", or carry the same information, almost perfectly.

In [ ]:
Credit = load_data('Credit')
print(f"correlation between Limit and Rating: {Credit['Limit'].corr(Credit['Rating']):.3f}")



`Limit` and `Rating` have a correlation coefficient of almost 1. Their "information overlap" is almost perfect.
Earlier we looked at what happens to model coefficients as we include more parameters. One case was low correlation, one case was moderate correlation, now let's look at almost perfect correlation:

In [ ]:
limit_model = smf.ols('Balance ~ Limit', data=Credit).fit()
limit_rating_model = smf.ols('Balance ~ Limit + Rating', data=Credit).fit()
display(compare_coefficient({'Limit alone': limit_model, 'Limit + Rating': limit_rating_model}, 'Limit'))
print(f"R²:  Limit alone = {limit_model.rsquared:.3f},  Limit + Rating = {limit_rating_model.rsquared:.3f}")

Adding `Rating` makes `Limit`'s standard error more than 10 times bigger, and its p-value jumps
from essentially 0 to about 0.7. The model can't tell which of two nearly identical predictors
deserves the credit, so very small changes in the sample can dramatically change how the credit
is split between them. The individual coefficients become unstable, and a real effect can look
non-significant. But $R^2$ barely changes: collinearity usually doesn't hurt **predictions**
much. The problem is **interpreting** individual coefficients.

The **variance inflation factor (VIF)** measures how much each coefficient's variance is
inflated by that predictor's correlation with the other predictors. A VIF near 1 means no
inflation. Values above roughly 5–10 are commonly treated as a concern, but that's a rule of
thumb, not a law.

In [ ]:
credit_model = smf.ols('Balance ~ Limit + Rating + Income + Age', data=Credit).fit()
X = credit_model.model.exog   # the predictor matrix statsmodels built, with a column of 1s for the intercept
pd.Series([variance_inflation_factor(X, i) for i in range(1, X.shape[1])],
          index=credit_model.model.exog_names[1:], name='VIF').round(2)

`Limit` and `Rating` are far past any threshold. `Income` is moderately correlated with both
($r \approx 0.79$), and `Age` is barely correlated with anything, so its VIF is close to 1. We'll
come back to collinearity later in the course, with regularization. For now, one reasonable approach with a correlation this high is to just drop one of the predictors (for example remove Rating), then fit a model on the remaining predictors.

## Part 5: Categorical predictors

Regression works with numbers, so a category becomes a set of **0/1 indicator columns**, one per
category value. The possible values of a category (here, `No` and `Yes`) are called **levels**. In statistics, this strategy is called **dummy variables**; in machine learning, it's usually called **one-hot encoding**. One level is left out as the **baseline**: with an intercept
in the model, its column would be redundant, since it's 1 exactly when all the others are 0.

In [ ]:
pd.concat([Carseats['US'],
           pd.get_dummies(Carseats['US'], prefix='US', drop_first=True, dtype=int)], axis=1).head()

You won't usually build these columns yourself. `statsmodels` creates them automatically for
categorical columns and lists each dummy as, for example, `US[T.Yes]`. (The `T` stands for
"treatment" coding, the default.) The level without its own row is the baseline.

**5(a) · Code.** Fit `Sales ~ Price + US`, name it `us_model`, and display its coefficients.

In [ ]:
# 5(a)
### YOUR CODE HERE


**5(b) · Written.** Which level of `US` is the baseline? How can you tell?

**5(b) Answer:**

_YOUR ANSWER HERE_


**5(c) · Written.** Interpret the `US[T.Yes]` coefficient in one sentence, with units.

**5(c) Answer:**

_YOUR ANSWER HERE_


For a two-level category, the dummy effectively shifts the whole line up or down (look at the multiple regression equation and think about why this is true). The result is two parallel
lines with the same `Price` slope. This cell uses your `us_model`. *(You don't need to understand
how this plot was generated.)*

In [ ]:
plot_parallel_lines(Carseats, 'Price', 'Sales', 'US', us_model);

**5(d) · Written *(optional)*.** `ShelveLoc` has three levels, so it gets two dummies. Here it's fitted for
you. Which level is the baseline, and what does the `ShelveLoc[T.Good]` coefficient mean?

In [ ]:
shelf_model = smf.ols('Sales ~ Price + ShelveLoc', data=Carseats).fit()
shelf_model.params

**5(d) Answer:**

_YOUR ANSWER HERE_


**The baseline isn't special.** It's just the reference point that the other levels are compared
with. Below, `Medium` is the baseline instead. The coefficients change, because they're now
differences from `Medium`, but every prediction stays the same. For example, Good vs. Medium is
$4.896 - 1.862 = 3.034$ in the first model, and $3.034$ directly in the second.

In [ ]:
shelf_medium_model = smf.ols("Sales ~ Price + C(ShelveLoc, Treatment('Medium'))", data=Carseats).fit()
display(shelf_medium_model.params)
print('Same predictions for every store:', np.allclose(shelf_model.fittedvalues, shelf_medium_model.fittedvalues))

## Part 6: Correlated error terms *(awareness)*

Regression's standard errors assume that each observation's error is independent of the others.
That breaks down in data collected over time (today's error tends to resemble yesterday's) or with
repeated measurements of the same subject. When errors are correlated, the standard errors come
out **too small**, so confidence intervals look narrower and p-values look smaller than they
should.

What it does **not** mean: the coefficient estimates aren't necessarily wrong. It's our
*confidence* in them that's overstated.